# Simulator Module Demo

Tests `OpenFOAMSimulator` and `OpenFOAMEnv` using a mock that replaces OpenFOAM with an analytical solution (Hagen-Poiseuille channel flow). No OpenFOAM installation needed.

In [1]:
%%capture
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import numpy as np
import xarray as xr
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import uqtopus as uqt

## Mock Simulator

Analytical channel flow: `U(y) = 6 * U_inlet * (y/H) * (1 - y/H)`, `p(x) = -12 * nu * U_inlet / H^2 * x`.
Parameter key follows UQTOPUS convention `folder__filename__paramname`.

In [3]:
class MockOpenFOAMSimulator:
    '''
    Drop-in replacement for OpenFOAMSimulator for testing without OpenFOAM.
    Simulates laminar channel flow (Hagen-Poiseuille) analytically.

    Parameters:
        n_cells (int)
            Number of cells in the spatial discretization. Default: 50.
        U_inlet (float)
            Fixed inlet velocity [m/s]. Default: 1.0
        H (float)
            Channel height [m]. Default: 0.1
        L (float)
            Channel length [m]. Default: 1.0
    '''

    PARAM_KEY = 'constant__physicalProperties__nu'

    def __init__(self, n_cells=50, U_inlet=1.0, H=0.1, L=1.0):
        self.n_cells = n_cells
        self.U_inlet = U_inlet
        self.H = H
        self.L = L
        self._run_count = 0

    @property
    def run_count(self):
        return self._run_count

    def run(self, params, step=None, verbose=False):
        nu = params.get(self.PARAM_KEY, 1e-5)
        y  = np.linspace(0, self.H, self.n_cells)
        x  = np.linspace(0, self.L, self.n_cells)

        U_x = 6 * self.U_inlet * (y / self.H) * (1.0 - y / self.H)
        U   = np.stack([U_x, np.zeros_like(U_x), np.zeros_like(U_x)], axis=-1)
        p   = -12 * nu * self.U_inlet / self.H ** 2 * x

        ds = xr.Dataset(
            {
                'U': xr.DataArray(U[np.newaxis], dims=['time', 'cell', 'component']),
                'p': xr.DataArray(p[np.newaxis], dims=['time', 'cell']),
            },
            coords={
                'time': [1.0],
                'x': ('cell', x),
                'y': ('cell', y),
                'z': ('cell', np.zeros(self.n_cells)),
            }
        )
        self._run_count += 1
        if verbose:
            Re     = self.U_inlet * self.H / nu
            p_drop = 12 * nu * self.U_inlet * self.L / self.H ** 2
            print(f'Run {self._run_count}: nu={nu:.2e}  Re={Re:.1f}  p_drop={p_drop:.4f} Pa')
        return ds

    def reset(self):
        self._run_count = 0

    def as_objective(self, metric_fn, param_keys):
        def objective(params_array):
            params = dict(zip(param_keys, np.asarray(params_array).tolist()))
            return metric_fn(self.run(params))
        return objective

    def as_residual(self, residual_fn, param_keys):
        def residual(params_array):
            params = dict(zip(param_keys, np.asarray(params_array).tolist()))
            return np.asarray(residual_fn(self.run(params)))
        return residual


PARAM_KEY = MockOpenFOAMSimulator.PARAM_KEY
sim = MockOpenFOAMSimulator()
sim

## 1. Single run

In [4]:
ds = sim.run({PARAM_KEY: 1e-4}, verbose=True)
ds

Run 1: nu=1.00e-04  Re=1000.0  p_drop=0.1200 Pa


<xarray.Dataset> Size: 3kB
Dimensions:  (time: 1, cell: 50, component: 3)
Coordinates:
  * time     (time) float64 8B 1.0
    x        (cell) float64 400B 0.0 0.02041 0.04082 ... 0.9592 0.9796 1.0
    y        (cell) float64 400B 0.0 0.002041 0.004082 ... 0.09592 0.09796 0.1
    z        (cell) float64 400B 0.0 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Dimensions without coordinates: cell, component
Data variables:
    U        (time, cell, component) float64 1kB 0.0 0.0 0.0 ... 0.0 0.0 0.0
    p        (time, cell) float64 400B -0.0 -0.002449 ... -0.1176 -0.12

## 2. Optimization with scipy.optimize

Find `nu` that achieves a target pressure drop of 0.5 Pa. `as_objective()` produces a plain callable — the loop stays in scipy.

In [5]:
from scipy.optimize import minimize_scalar

TARGET_P_DROP = 0.5  # Pa

def metric_fn(ds):
    p_drop = abs(float(ds['p'].values[0, -1] - ds['p'].values[0, 0]))
    return (p_drop - TARGET_P_DROP) ** 2

sim.reset()
f = sim.as_objective(metric_fn=metric_fn, param_keys=[PARAM_KEY])

result = minimize_scalar(lambda nu: f([nu]), bounds=(1e-5, 1e-2), method='bounded')

nu_opt  = result.x
p_achieved = abs(float(sim.run({PARAM_KEY: nu_opt})['p'].values[0, -1]))

print(f'Target:   {TARGET_P_DROP:.4f} Pa')
print(f'Optimal nu: {nu_opt:.4e} m^2/s')
print(f'Achieved: {p_achieved:.4f} Pa')
print(f'Runs: {sim.run_count}')

Target:   0.5000 Pa
Optimal nu: 4.1667e-04 m^2/s
Achieved: 0.5000 Pa
Runs: 8


## 3. Parameter fitting with scipy.optimize.least_squares

Generate noisy velocity measurements with a known `nu`, then recover it.

In [6]:
from scipy.optimize import least_squares

nu_true = 3e-4
sim.reset()
target = sim.run({PARAM_KEY: nu_true})['U'].values[0, :, 0].copy()
target += np.random.default_rng(42).normal(0, 0.005, sim.n_cells)  # measurement noise

r = sim.as_residual(
    residual_fn=lambda ds: ds['U'].values[0, :, 0] - target,
    param_keys=[PARAM_KEY]
)

result = least_squares(r, x0=[1e-3], bounds=([1e-5], [1e-2]))

print(f'True nu:      {nu_true:.4e}')
print(f'Recovered nu: {result.x[0]:.4e}')
print(f'Runs: {sim.run_count}')

True nu:      3.0000e-04
Recovered nu: 1.0000e-03
Runs: 3


## 4. RL environment with gymnasium

Agent controls `nu`, reward = -|p_drop - target_p_drop|.

In [7]:
from uqtopus.envs import OpenFOAMEnv

TARGET_RL = 0.2  # Pa

def obs_fn(ds):
    U_max  = float(ds['U'].values[0, :, 0].max())
    p_drop = abs(float(ds['p'].values[0, -1] - ds['p'].values[0, 0]))
    return np.array([U_max, p_drop], dtype=np.float32)

def reward_fn(ds):
    p_drop = abs(float(ds['p'].values[0, -1] - ds['p'].values[0, 0]))
    return -abs(p_drop - TARGET_RL)

sim.reset()

env = OpenFOAMEnv(
    simulator=sim,
    param_ranges={PARAM_KEY: (1e-5, 1e-2)},
    observation_fn=obs_fn,
    reward_fn=reward_fn,
    obs_shape=(2,),
    max_episode_steps=10,
    initial_params={PARAM_KEY: 1e-3},
)

print(env)
print('action_space:', env.action_space)
print('observation_space:', env.observation_space)

<OpenFOAMEnv instance>
action_space: Box(1e-05, 0.01, (1,), float32)
observation_space: Box(-inf, inf, (2,), float32)


Manual episode with random actions

In [8]:
obs, info = env.reset()
print(f'reset  obs={obs}  params={info["params"]}')

for step in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    nu_val = info['params'][PARAM_KEY]
    print(f'step {step + 1}  nu={nu_val:.2e}  reward={reward:.4f}  obs={obs}')
    if terminated or truncated:
        break

reset  obs=[1.4993752 1.2      ]  params={'constant__physicalProperties__nu': 0.001}
step 1  nu=1.62e-03  reward=-1.7458  obs=[1.4993752 1.945844 ]
step 2  nu=6.06e-03  reward=-7.0731  obs=[1.4993752 7.273058 ]
step 3  nu=3.22e-03  reward=-3.6601  obs=[1.4993752 3.8601053]
step 4  nu=9.86e-04  reward=-0.9834  obs=[1.4993752 1.183442 ]
step 5  nu=6.05e-03  reward=-7.0605  obs=[1.4993752 7.26052  ]


## 5. Training with stable-baselines3

The environment is fully SB3-compatible. To train:

In [9]:
from stable_baselines3 import PPO
sim.reset()
model = PPO('MlpPolicy', env, verbose=1)
model.learn(total_timesteps=200)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 10       |
|    ep_rew_mean     | -60.7    |
| time/              |          |
|    fps             | 422      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------


To switch to a real OpenFOAM run, replace `MockOpenFOAMSimulator` with `OpenFOAMSimulator` and point it at your template. Everything else stays the same.